# 03-3 · The lambda dial — does it move the pick?

| | |
|---|---|
| **What it does** | sweeps lambda from 0 to 100 on the real catalog, then on a constructed catalog designed to conflict |
| **Proves / disproves** | Figure 1's claim only holds where mean and variance actually trade off — measured here, not assumed |
| **Uses** | `nbio.fairness_score`, `nbio.aggregate` |
| **Eval surface** | the crossing point where `fair(lambda)` stops agreeing with `average` |

No API key needed.


In [1]:
# Locate the repo root (nbio.py lives there) regardless of where this notebook sits.
import os, sys
_root = os.getcwd()
for _ in range(4):
    if os.path.isfile(os.path.join(_root, "nbio.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)


In [2]:
import nbio, json
nbio.bootstrap()


repo     : /project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-food/research_fair-mediator
data     : data  (ingredient-flavor.csv: found, 5 menu files)


PosixPath('/project/painting/project-research/AnacodicAI-Labs/code/production-repos/ai-in-food/research_fair-mediator')

## Rebuild F, S on the real catalog (same 4 members as 03-1/03-2)


In [3]:
# Step 1: rebuild
personas = json.load(open(nbio.SMALL_PERSONAS))
group_names = ["Ravi", "Marco", "Omar", "Alice"]
members = []
for name in group_names:
    p = next(x for x in personas if x["display_name"] == name)
    members.append({"member_id": name.lower(), "display_name": name,
                     "diet": p["diet_type"], "allergies": p["allergies"], "taste": p["taste_vector"]})

import csv
flavor_map = {}
with open(nbio.DATA / "ingredient-flavor.csv") as f:
    for row in csv.DictReader(f):
        n = row.get("ingredient", "").strip().lower()
        if n: flavor_map[n] = {d: float(row.get(d, 0) or 0) for d in nbio.TASTE_DIMS}

def iter_dish_names(node):
    if isinstance(node, dict):
        if ("item" in node or "name" in node) and "price" in node:
            yield (node.get("item") or node.get("name") or "").strip(); return
        for v in node.values(): yield from iter_dish_names(v)
    elif isinstance(node, list):
        for v in node: yield from iter_dish_names(v)

def infer_taste(d):
    m = [flavor_map[i] for i in flavor_map if i in d.lower()]
    return [sum(x[k] for x in m)/len(m) for k in nbio.TASTE_DIMS] if m else [0.0]*6

catalog = []
for f in sorted(nbio.SMALL_MENUS.glob("*.json")):
    names = [n for n in iter_dish_names(json.loads(f.read_text())) if n]
    vecs = [infer_taste(n) for n in names]
    nz = [v for v in vecs if sum(abs(x) for x in v) > 0]
    avg = [sum(v[i] for v in nz)/len(nz) for i in range(6)] if nz else [0.0]*6
    catalog.append({"option_id": f.stem, "name": f.stem, "menu": [{"name": n} for n in names], "taste": avg})

F = nbio.build_feasible_set(catalog, members)
candidate_vectors = {o["option_id"]: o["taste"] for o in F}
member_vectors = [m["taste"] for m in members]
labels = {o["option_id"]: o["name"] for o in F}
print(f"F size: {len(F)}")


F size: 5


## Sweep lambda on the real catalog


In [4]:
# Step 2: does raising lambda ever change the pick?
seen = {}
for lam in [0, 0.5, 1, 2, 5, 10, 25, 50, 100]:
    ranked = nbio.rank_candidates(candidate_vectors, member_vectors, method="fair", lam=lam)
    winner_id, score = ranked[0]
    seen.setdefault(labels[winner_id], []).append(lam)
    print(f"   lambda={lam:<5} -> {labels[winner_id]:<32} score={score:+.4f}")

print()
print("distinct picks across lambda:", seen)


   lambda=0     -> Casa.cafe                        score=+0.7609
   lambda=0.5   -> Casa.cafe                        score=+0.7571
   lambda=1     -> Casa.cafe                        score=+0.7533
   lambda=2     -> Casa.cafe                        score=+0.7456
   lambda=5     -> Casa.cafe                        score=+0.7225
   lambda=10    -> Casa.cafe                        score=+0.6840
   lambda=25    -> Delicia Multi Cuisine Restaurant score=+0.5709
   lambda=50    -> Delicia Multi Cuisine Restaurant score=+0.3832
   lambda=100   -> Delicia Multi Cuisine Restaurant score=+0.0077

distinct picks across lambda: {'Casa.cafe': [0, 0.5, 1, 2, 5, 10], 'Delicia Multi Cuisine Restaurant': [25, 50, 100]}


> **Measured on this 4-member group: the pick DOES move, but only at a large
> lambda.** `fair(lambda)` agrees with `average` (picks Casa.cafe) up to
> lambda=10, then crosses over to Delicia Multi Cuisine at lambda=25 and stays
> there through 100. The dial is not inert here — it just needs an unusually
> large fairness penalty to overturn the mean-maximiser, which itself is a
> measurement: the mean and variance objectives are only weakly in tension on
> this catalog. A catalog with more genuine spread should show the
> crossover at a smaller, more interpretable lambda.


## Does the mechanism itself work? A constructed conflict

Three diners in the shape of the paper's own worked example: two want heat, one
wants umami, and — this is the deliberate design choice — **no balanced option
exists**. With cosine similarity a balanced option would dominate on mean AND
minimum simultaneously, so a genuine conflict requires its absence.


In [5]:
# Step 3: a catalog built to make Average and Least-Misery disagree
toy_members = [
    {"member_id": "ann",  "display_name": "Ann",  "taste": [0, 0, 0, 0, 0.0, 1.0]},
    {"member_id": "ben",  "display_name": "Ben",  "taste": [0, 0, 0, 0, 0.2, 0.98]},
    {"member_id": "cara", "display_name": "Cara", "taste": [0, 0, 0, 0, 1.0, 0.0]},
]
toy_catalog = {
    "chilli_house": [0, 0, 0, 0, 0.0, 1.0],
    "umami_den":    [0, 0, 0, 0, 0.95, 0.31],
    "sweet_stop":   [1.0, 0, 0, 0, 0.0, 0.0],
}
toy_member_vecs = [m["taste"] for m in toy_members]

seen = {}
for lam in [0, 0.25, 0.5, 0.75, 1, 2, 4]:
    ranked = nbio.rank_candidates(toy_catalog, toy_member_vecs, method="fair", lam=lam)
    winner, score = ranked[0]
    seen.setdefault(winner, []).append(lam)
    print(f"   lambda={lam:<5} -> {winner:<14} score={score:+.4f}")
print()
print("lambda -> winner:", seen)


   lambda=0     -> chilli_house   score=+0.6599
   lambda=0.25  -> chilli_house   score=+0.6055
   lambda=0.5   -> chilli_house   score=+0.5510
   lambda=0.75  -> umami_den      score=+0.5306
   lambda=1     -> umami_den      score=+0.5125
   lambda=2     -> umami_den      score=+0.4400
   lambda=4     -> umami_den      score=+0.2950

lambda -> winner: {'chilli_house': [0, 0.25, 0.5], 'umami_den': [0.75, 1, 2, 4]}


In [6]:
# Step 4: assert the machinery genuinely responds to a real conflict
avg_pick = nbio.rank_candidates(toy_catalog, toy_member_vecs, method="average")[0][0]
lm_pick  = nbio.rank_candidates(toy_catalog, toy_member_vecs, method="least_misery")[0][0]
lo_pick  = nbio.rank_candidates(toy_catalog, toy_member_vecs, method="fair", lam=0.0)[0][0]
hi_pick  = nbio.rank_candidates(toy_catalog, toy_member_vecs, method="fair", lam=4.0)[0][0]

assert avg_pick != lm_pick, "constructed case failed to create a selector conflict"
assert lo_pick != hi_pick, "constructed case failed to make lambda move the pick"
print(f"PASS -- average picks {avg_pick!r}; least_misery picks {lm_pick!r}")
print(f"PASS -- fair(0) picks {lo_pick!r}; fair(4) picks {hi_pick!r}")


PASS -- average picks 'chilli_house'; least_misery picks 'umami_den'
PASS -- fair(0) picks 'chilli_house'; fair(4) picks 'umami_den'


### Reading it

Two findings, and they are not in tension. **(1)** On the real catalog, `lambda`
does move the pick, but only once it crosses roughly 10-25 — a much larger
penalty than a reader would expect to matter, because five real restaurants
here are similar enough that the mean and variance objectives only weakly
disagree. **(2)** On a catalog constructed to contain a genuine trade-off,
`lambda` moves the pick at a small, interpretable value (between 0.5 and
0.75). Both assertions in Step 4 pass. **Consequence for Figure 1**: the crossover
point itself is a number worth reporting — it measures how much fairness
pressure this catalog needs before it matters — but a curve drawn only from this
5-restaurant catalog will look flatter than the true shape of the trade-off.
Larger menus and synthetic groups with real conflicts are what let Figure 1
show the crossover at a realistic scale.


## C1 on real restaurants: how far does the dial move the worst-off diner?

The harness (`06-evaluate/01`) reports the lambda sweep; this section computes
the C1 numbers it does not print directly, with the same functions and the same
200 seeded groups (seed 7, 35% adversarial), for the small catalog and for Uber
Eats Houston and Seattle, each also with shuffled menus (the random-menu control).

For every dataset it reports:

- **moved**: % of groups where the pick at lambda 0 differs from lambda 5 (and from any lambda in the sweep);
- **gain**: per-group MinSat(Fair) - MinSat(Average), mean and a 95% bootstrap CI over groups, with the mean-satisfaction cost;
- **achieved**: the gain as a % of the achievable gap, Least-Misery's MinSat minus Average's, i.e. the most any option could lift the worst-off diner (ratio of sums, bootstrap CI over groups);
- under both satisfaction formulas: **A** whole menu, `s_i(x) = cos(t_i, mean taste of all dishes at x)`, and **B** safe subset, `s_i(x) = cos(t_i, mean taste of i's safe dishes at x)`.

Groups share members drawn from one diner pool, so they are not fully
independent; the CI is over groups and is optimistic by that much. The helper
functions are `nbio.c1_*` (`02-safety/01` and `06-evaluate/02` reuse them).

Writes `c1_summary.csv` and `table3_datasets.csv` (Fair vs Average for both
cities, formula A, lambda 0.5) into each folder in `C1_OUT_DIRS`. Takes a few
minutes (pure-Python bootstrap).

In [7]:
# Parameters
C1_PERSONAS = "foodcom"              # diner pool: Food.com users (FM_PERSONAS)
C1_OUT_DIRS = ["figdata/ubereats", "figdata/ubereats-foodcom"]

In [8]:
os.environ["FM_PERSONAS"] = C1_PERSONAS
c1_rows, c1_table3 = [], []
for label, dataset, city in nbio.C1_DATASETS:
    print(f"... {label}", flush=True)
    rows_, per_ = nbio.c1_evaluate(label, dataset, city)
    c1_rows += rows_
    if dataset == "ubereats":
        c1_table3 += nbio.c1_table3_rows(label, per_["A"])
for d in C1_OUT_DIRS:
    os.environ["FM_FIGDATA"] = d
    nbio.save_figdata("c1_summary.csv", c1_rows)
    nbio.save_figdata("table3_datasets.csv", c1_table3)
os.environ.pop("FM_FIGDATA")
print()
nbio.table(c1_rows, cols=["dataset", "formula", "lambda", "pct_moved_0_vs_5", "minsat_gain",
                          "minsat_gain_ci_lo", "minsat_gain_ci_hi", "meansat_change"], width=16)
print()
nbio.table(c1_table3, cols=["dataset", "selector", "mean_sat", "min_sat", "jain"], width=22)

... small catalog (5 restaurants)


... Houston, TX


... Houston, TX shuffled


... Seattle, WA


... Seattle, WA shuffled


-> wrote figdata/ubereats/c1_summary.csv  (20 rows)
-> wrote figdata/ubereats/table3_datasets.csv  (4 rows)
-> wrote figdata/ubereats-foodcom/c1_summary.csv  (20 rows)
-> wrote figdata/ubereats-foodcom/table3_datasets.csv  (4 rows)

dataset           formula      lambda  pct_moved_0_vs_5  minsat_gain  minsat_gain_ci_l  minsat_gain_ci_h  meansat_change
------------------------------------------------------------------------------------------------------------------------
small catalog (5  whole menu   0.500   3.500             0.001        0.000             0.002             -0.000        
small catalog (5  whole menu   5       3.500             0.002        0.000             0.004             -0.000        
small catalog (5  safe subset  0.500   6.500             0.000        0.000             0.000             0.000         
small catalog (5  safe subset  5       6.500             0.002        0.001             0.003             -0.000        
Houston, TX       whole menu   0.500   41

## The theory checked on real diners (Propositions 1-4)

Food.com diners on Uber Eats Houston and Seattle; the same groups, feasible sets,
satisfaction matrix and pick rule as the C1 section above (whole-menu formula A;
pick = first maximum of `nbio.aggregate` over F). For every lambda in the
harness sweep and every group it checks:

- **dominated**: is the Fair pick Pareto-dominated inside F (some option weakly better for every diner, strictly for one)? (Prop. 1)
- **threshold**: `lambda*(n) = n / (2(n-1))`, the exact bound on `[0,1]^n`, and `lambda*_data = n / (2(n-1) R)` with R the range of the group's S. Below either bound Prop. 1 says 0 dominated; the counts are reported.
- **loss / bound**: `mean(x_avg) - mean(x_fair)` against `lambda (Var(x_avg) - Var(x_fair))`, `lambda Var(x_avg)`, `lambda max_F Var` and `lambda/4` (Prop. 2)
- **minsat**: `MinSat(x_fair) < MinSat(x_avg)`? (Prop. 3)
- **monotone**: Var and mean of the Fair pick non-increasing in lambda (Prop. 4)

plus the mediator (`nbio.mediate`, lambda 0.5, the paper's settings): is its
final pick dominated, and is its weighted threshold `n^2 w_min / (2(n-1))` met?

Writes `figdata/ubereats-foodcom/pareto_check.csv`: rows per (dataset, group
size, lambda), group size "all" first.

In [9]:
SEED, N_GROUPS, ADVERSARIAL_FRAC = 7, 200, 0.35       # as in 06-evaluate
SWEEP = [0, 0.25, 0.5, 1, 2, 5, 10, 25, 50, 100]       # as in 06-evaluate
CITIES = ["Houston, TX", "Seattle, WA"]
OUT = nbio.HERE / "figdata" / "ubereats-foodcom"
TOL = 1e-12


def var(v):
    return nbio.variance(v)


def dominated(S, pick, mids):
    """True if some option in F is >= for every diner and > for at least one."""
    p = [S[pick][m] for m in mids]
    for o, row in S.items():
        if o == pick:
            continue
        q = [row[m] for m in mids]
        if all(a >= b for a, b in zip(q, p)) and any(a > b for a, b in zip(q, p)):
            return True
    return False


pick = nbio.c1_pick      # first maximum of nbio.aggregate, as in the C1 section


def per_group(catalog, groups):
    out = []
    for g in groups:
        members = g["members"]
        mids = [m["member_id"] for m in members]
        F = nbio.build_feasible_set(catalog, members)
        if not F:
            continue
        S = {o["option_id"]: {m["member_id"]: nbio.taste_similarity(m["taste"], o["taste"])
                              for m in members} for o in F}
        n = len(mids)
        vals = [s for row in S.values() for s in row.values()]
        R = max(vals) - min(vals)
        avg = pick(S, "average")
        va = [S[avg][m] for m in mids]
        rec = {
            "n": n, "F": len(S), "R": R,
            "lam_star_n": n / (2 * (n - 1)),
            "lam_star_data": n / (2 * (n - 1) * R) if R > 0 else float("inf"),
            "max_var": max(var(list(row.values())) for row in S.values()),
            "any_dominated_option": any(dominated(S, o, mids) for o in S),
            "avg": va, "fair": {},
        }
        for lam in SWEEP:
            f = pick(S, "fair", lam)
            rec["fair"][lam] = {"vals": [S[f][m] for m in mids], "dom": dominated(S, f, mids)}
        med = nbio.mediate(S, mids, lam=0.5)
        if med["pick"] is not None:
            last = med["history"][-1]
            wmin = min(last["weights"].values())
            rec["med"] = {"dom": dominated(S, last["pick"], mids),
                          "lam_star_w": n * n * wmin / (2 * (n - 1)), "wmin": wmin}
        out.append(rec)
    return out


def rows_for(label, recs, size):
    gs = recs if size == "all" else [r for r in recs if r["n"] == size]
    if not gs:
        return []
    rows = []
    prev = None
    for lam in SWEEP:
        dom = [r["fair"][lam]["dom"] for r in gs]
        below_n = [r for r in gs if lam <= r["lam_star_n"]]
        below_d = [r for r in gs if lam <= r["lam_star_data"]]
        loss, b_gap, b_avg, b_max = [], [], [], []
        viol = drop = 0
        for r in gs:
            vf, va = r["fair"][lam]["vals"], r["avg"]
            L = nbio.mean_sat(va) - nbio.mean_sat(vf)
            bg = lam * (var(va) - var(vf))
            loss.append(L); b_gap.append(bg)
            b_avg.append(lam * var(va)); b_max.append(lam * r["max_var"])
            viol += (L > bg + TOL) or (bg > lam * var(va) + TOL) or (L < -TOL)
            drop += min(vf) < min(va) - TOL
        mono_var = mono_mean = 0
        if prev is not None:
            for r in gs:
                a, b = r["fair"][prev]["vals"], r["fair"][lam]["vals"]
                mono_var += var(b) > var(a) + TOL
                mono_mean += nbio.mean_sat(b) > nbio.mean_sat(a) + TOL
        k = len(gs)
        rows.append({
            "dataset": label, "group_size": size, "lambda": lam, "n_groups": k,
            "lambda_star_n": round(gs[0]["lam_star_n"], 4) if size != "all" else "",
            "n_dominated": sum(dom), "pct_dominated": round(100 * sum(dom) / k, 1),
            "n_below_lambda_star_n": len(below_n),
            "n_dominated_below_lambda_star_n": sum(r["fair"][lam]["dom"] for r in below_n),
            "n_below_lambda_star_data": len(below_d),
            "n_dominated_below_lambda_star_data": sum(r["fair"][lam]["dom"] for r in below_d),
            "pct_groups_with_any_dominated_option": round(100 * sum(r["any_dominated_option"] for r in gs) / k, 1),
            "meansat_loss_mean": round(sum(loss) / k, 6),
            "meansat_loss_max": round(max(loss), 6),
            "bound_var_gap_mean": round(sum(b_gap) / k, 6),
            "bound_var_avg_mean": round(sum(b_avg) / k, 6),
            "bound_max_var_F_mean": round(sum(b_max) / k, 6),
            "bound_quarter": round(lam / 4, 6),
            "n_p2_violations": viol,
            "n_minsat_drop": drop, "pct_minsat_drop": round(100 * drop / k, 1),
            "n_var_increase_vs_prev_lambda": mono_var if prev is not None else "",
            "n_mean_increase_vs_prev_lambda": mono_mean if prev is not None else "",
        })
        prev = lam
    return rows


def mediator_row(label, recs):
    ms = [r["med"] for r in recs if "med" in r]
    below = [m for m in ms if 0.5 <= m["lam_star_w"]]
    return {"dataset": label, "group_size": "all", "lambda": "mediator 0.5", "n_groups": len(ms),
            "n_dominated": sum(m["dom"] for m in ms),
            "pct_dominated": round(100 * sum(m["dom"] for m in ms) / len(ms), 1),
            "n_below_lambda_star_n": len(below),
            "n_dominated_below_lambda_star_n": sum(m["dom"] for m in below),
            "lambda_star_n": round(min(m["lam_star_w"] for m in ms), 6)}

In [10]:
os.environ["FM_PERSONAS"] = "foodcom"
os.environ["FM_DATASET"] = "ubereats"
rows, meds = [], []
for city in CITIES:
    print(f"... {city}", flush=True)
    os.environ["FM_CITY"] = city
    catalog = nbio.load_catalog()
    groups = nbio.generate_groups(n_groups=N_GROUPS, seed=SEED, adversarial_frac=ADVERSARIAL_FRAC)
    recs = per_group(catalog, groups)
    for size in ["all", 3, 4, 5, 6]:
        rows += rows_for(city, recs, size)
    meds.append(mediator_row(city, recs))
    R = [r["R"] for r in recs]
    print(f"    groups {len(recs)}  S range per group: min {min(R):.3f} median "
          f"{sorted(R)[len(R) // 2]:.3f} max {max(R):.3f}", flush=True)
rows += meds
cols = list(rows[0].keys())
rows = [{c: r.get(c, "") for c in cols} for r in rows]
OUT.mkdir(parents=True, exist_ok=True)
os.environ["FM_FIGDATA"] = str(OUT.relative_to(nbio.HERE))
nbio.save_figdata("pareto_check.csv", rows)
os.environ.pop("FM_FIGDATA")
nbio.table([r for r in rows if r["group_size"] == "all"],
           cols=["dataset", "lambda", "n_groups", "n_dominated", "pct_dominated",
                 "n_dominated_below_lambda_star_n", "meansat_loss_mean", "bound_var_avg_mean",
                 "n_p2_violations", "pct_minsat_drop"], width=14)

... Houston, TX


    groups 200  S range per group: min 0.246 median 0.460 max 0.707


... Seattle, WA


    groups 200  S range per group: min 0.280 median 0.613 max 0.815


-> wrote figdata/ubereats-foodcom/pareto_check.csv  (102 rows)
dataset      lambda        n_groups  n_dominated  pct_dominated  n_dominated_be  meansat_loss_m  bound_var_avg_  n_p2_violation  pct_minsat_dro
-----------------------------------------------------------------------------------------------------------------------------------------------
Houston, TX  0             200       0            0.000          0               0.000           0.000           0               0.000         
Houston, TX  0.250         200       0            0.000          0               0.000           0.001           0               0.000         
Houston, TX  0.500         200       0            0.000          0               0.000           0.002           0               0.000         
Houston, TX  1             200       0            0.000          0               0.000           0.003           0               0.500         
Houston, TX  2             200       0            0.000          0       

## Revision: why lambda = 5 is safe on these data

Three analyses on the per-group matrices (`nbio.rev_data()`, Food.com diners on
Uber Eats Houston and Seattle):

1. **E6**: per group, the exact critical lambda (the smallest lambda at which some
   dominated option can outscore one of its dominators), the range of S over all
   options and over the options Fair could pick for lambda <= 5, the first lambda
   at which the Fair pick moves, the Pareto-front size, and how often a
   Pareto-filtered Fair rule picks differently from plain Fair over the sweep.
   Writes `figdata_rev/e6_lambda.csv` (per city) and `e6_lambda_groups.csv` (per group).
2. **E6b**: the first lambda on a log grid from 1 to 1000 at which the Fair pick is
   Pareto-dominated. Writes `figdata_rev/e6_first_dominated.csv`.
3. **E6c**: how many groups the data-dependent bound `n / (2(n-1) R)` certifies at
   lambda 1, 2, 5, 10 and 25. Writes `figdata_rev/e6_certificate.csv`.

The last cell writes the LaTeX table rows `figdata_rev/tex/tab_lambda.tex` from
`e6_lambda.csv` and `e6_first_dominated.csv`.

In [11]:
import csv
import numpy as np
from nbio import REV_CITIES, REV_LAMBDAS, rev_data, rev_pick

OUT = nbio.REV_OUT
D = rev_data()


def crit_lambda(S):
    m = S.mean(1); v = S.var(1)
    best = np.inf
    # y dominates x: all S[y] >= S[x], some >
    for x in range(len(S)):
        ge = np.all(S >= S[x], axis=1) & np.any(S > S[x], axis=1)
        ys = np.where(ge & (v > v[x]))[0]
        if len(ys):
            best = min(best, np.min((m[ys] - m[x]) / (v[ys] - v[x])))
    return best


def pareto_front(S):
    keep = np.ones(len(S), bool)
    for x in range(len(S)):
        if np.any(np.all(S >= S[x], axis=1) & np.any(S > S[x], axis=1)):
            keep[x] = False
    return keep

In [12]:
# E6
rows, grows = [], []
for city in REV_CITIES:
    crits, Rs, Rtop, cross, fronts = [], [], [], [], []
    for g in D[city]:
        S = g["S"]
        lc = crit_lambda(S); crits.append(lc)
        Rs.append(S.max() - S.min())
        # range over the options Fair could pick for lambda<=5 (price-of-fairness set)
        a = rev_pick(S, "average")
        cand = S.mean(1) >= S[a].mean() - 5 * S.var(1).max()
        Rtop.append(S[cand].max() - S[cand].min())
        a0 = rev_pick(S, "fair", 0)
        moved = [lam for lam in np.logspace(-2, 3, 201) if rev_pick(S, "fair", lam) != a0]
        cross.append(moved[0] if moved else np.inf)
        fr = pareto_front(S); fronts.append(fr)
        grows.append(dict(city=city, group_id=g["group_id"], n=g["n"], lambda_crit=round(lc, 3),
                          range_all=round(Rs[-1], 4), first_move=round(cross[-1], 3),
                          front_size=int(fr.sum()), F_size=len(S)))
    crits = np.array(crits); cross = np.array(cross)
    # Pareto-filtered Fair vs Fair over the sweep
    diff = {}
    for lam in REV_LAMBDAS:
        diff[lam] = sum(rev_pick(g["S"], "fair", lam) != rev_pick(g["S"], "fair", lam, mask=fr)
                        for g, fr in zip(D[city], fronts))
    finite = cross[np.isfinite(cross)]
    rows.append(dict(city=city,
                     crit_min=round(crits.min(), 2), crit_p5=round(np.percentile(crits, 5), 2),
                     crit_median=round(np.median(crits), 2),
                     groups_crit_le_5=int((crits <= 5).sum()), groups_crit_le_25=int((crits <= 25).sum()),
                     range_all_median=round(np.median(Rs), 3),
                     range_competitive_median=round(np.median(Rtop), 3),
                     front_size_median=int(np.median([f.sum() for f in fronts])),
                     first_move_median=round(np.median(finite), 2), groups_never_move=int((~np.isfinite(cross)).sum()),
                     **{f"pf_differs_lam{lam}": diff[lam] for lam in REV_LAMBDAS}))
    print(rows[-1])

for name, rr in [("e6_lambda.csv", rows), ("e6_lambda_groups.csv", grows)]:
    with open(f"{OUT}/{name}", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rr[0].keys())); w.writeheader(); w.writerows(rr)

{'city': 'Houston, TX', 'crit_min': np.float64(1.57), 'crit_p5': np.float64(1.79), 'crit_median': np.float64(12.13), 'groups_crit_le_5': 53, 'groups_crit_le_25': 174, 'range_all_median': np.float64(0.46), 'range_competitive_median': np.float64(0.21), 'front_size_median': 53, 'first_move_median': np.float64(7.08), 'groups_never_move': 9, 'pf_differs_lam0': 0, 'pf_differs_lam0.25': 0, 'pf_differs_lam0.5': 0, 'pf_differs_lam1': 0, 'pf_differs_lam2': 0, 'pf_differs_lam5': 0, 'pf_differs_lam10': 0, 'pf_differs_lam25': 0, 'pf_differs_lam50': 2, 'pf_differs_lam100': 16}


{'city': 'Seattle, WA', 'crit_min': np.float64(1.79), 'crit_p5': np.float64(1.97), 'crit_median': np.float64(11.97), 'groups_crit_le_5': 52, 'groups_crit_le_25': 174, 'range_all_median': np.float64(0.608), 'range_competitive_median': np.float64(0.234), 'front_size_median': 55, 'first_move_median': np.float64(6.31), 'groups_never_move': 11, 'pf_differs_lam0': 0, 'pf_differs_lam0.25': 0, 'pf_differs_lam0.5': 0, 'pf_differs_lam1': 0, 'pf_differs_lam2': 0, 'pf_differs_lam5': 0, 'pf_differs_lam10': 0, 'pf_differs_lam25': 0, 'pf_differs_lam50': 4, 'pf_differs_lam100': 7}


In [13]:
# E6b: first lambda at which the Fair pick is Pareto-dominated
grid = np.logspace(0, 3, 121)
rows=[]
for city in REV_CITIES:
    firsts=[]
    for g in D[city]:
        S=g["S"]; f=np.inf
        for lam in grid:
            p=rev_pick(S,"fair",lam)
            if np.any(np.all(S>=S[p],axis=1)&np.any(S>S[p],axis=1)): f=lam; break
        firsts.append(f)
    firsts=np.array(firsts); fin=firsts[np.isfinite(firsts)]
    rows.append(dict(city=city, first_dominated_min=round(fin.min(),1), first_dominated_p5=round(np.percentile(firsts,5),1),
          groups_ever_dominated_le1000=len(fin), groups_dominated_le_10=int((firsts<=10).sum()), groups_dominated_le_25=int((firsts<=25).sum())))
    print(rows[-1])
with open(OUT + "/e6_first_dominated.csv","w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)

{'city': 'Houston, TX', 'first_dominated_min': np.float64(26.6), 'first_dominated_p5': np.float64(93.9), 'groups_ever_dominated_le1000': 69, 'groups_dominated_le_10': 0, 'groups_dominated_le_25': 0}


{'city': 'Seattle, WA', 'first_dominated_min': np.float64(37.6), 'first_dominated_p5': np.float64(111.9), 'groups_ever_dominated_le1000': 87, 'groups_dominated_le_10': 0, 'groups_dominated_le_25': 0}


In [14]:
# E6c: groups certified by the data-dependent bound
rows=[]
for city in REV_CITIES:
    out={}
    for lam in (1,2,5,10,25):
        cert=0
        for g in D[city]:
            S=g["S"]; n=g["n"]; ls=n/(2*(n-1))
            a=rev_pick(S,"average"); C=S.mean(1)>=S[a].mean()-lam*S[a].var()
            R=S[C].max()-S[C].min()
            cert+= lam <= ls/R if R>0 else True
        out[lam]=cert
    rows.append(dict(city=city, **{f"certified_lam{l}":v for l,v in out.items()}))
    print(rows[-1])
with open(OUT + "/e6_certificate.csv","w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)

{'city': 'Houston, TX', 'certified_lam1': np.int64(200), 'certified_lam2': np.int64(170), 'certified_lam5': np.int64(131), 'certified_lam10': np.int64(95), 'certified_lam25': np.int64(42)}
{'city': 'Seattle, WA', 'certified_lam1': np.int64(200), 'certified_lam2': np.int64(185), 'certified_lam5': np.int64(134), 'certified_lam10': np.int64(98), 'certified_lam25': np.int64(36)}


In [15]:
# LaTeX table rows (tab_lambda.tex) from the CSVs above
import pandas as pd
R = nbio.REV_OUT
T = os.path.join(nbio.REV_OUT, "tex")
os.makedirs(T, exist_ok=True)
H, S = "Houston, TX", "Seattle, WA"
l = pd.read_csv(f"{R}/e6_lambda.csv"); fd = pd.read_csv(f"{R}/e6_first_dominated.csv")
lines = []
for c in (H, S):
    r = l[l.city == c].iloc[0]; d = fd[fd.city == c].iloc[0]
    lines.append(" & ".join([c.split(",")[0], f"{r.first_move_median:.2f}", f"{r.range_all_median:.2f}",
                             f"{r.range_competitive_median:.2f}", f"{d.first_dominated_min:.1f}",
                             f"{int(r.pf_differs_lam25)}", f"{int(r.pf_differs_lam50)}", f"{int(r.pf_differs_lam100)}"]) + r" \\")
open(f"{T}/tab_lambda.tex", "w").write("\n".join(lines) + "\n")
print(open(f"{T}/tab_lambda.tex").read())

Houston & 7.08 & 0.46 & 0.21 & 26.6 & 0 & 2 & 16 \\
Seattle & 6.31 & 0.61 & 0.23 & 37.6 & 0 & 4 & 7 \\

